# 5단계: 피클 파일 생성 및 통합 테스트

## 목표
- 프로덕션용 피클 파일 생성
- 모델 로더 클래스 구현
- API 통합 테스트
- 성능 벤치마크

In [10]:
# 필수 라이브러리 임포트
import pandas as pd
import numpy as np
import pickle
import json
import os
import time
from collections import defaultdict
from typing import List, Dict, Set, Optional, Any
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

print("라이브러리 로딩 완료")

라이브러리 로딩 완료


## 5.1 프로덕션용 모델 클래스 정의

In [11]:
class RecipeGapFillingModel:
    """
    프로덕션용 레시피 GapFilling 추천 모델 v2.0
    
    장바구니에 담긴 재료를 분석하여:
    1. 만들 수 있는 레시피를 추천 (요리명 CKG_NM 사용)
    2. 부족한 재료(Gap)를 식별하여 추가 상품 추천
    3. 메인 재료 우선 추천 (육류/해산물 > 양념)
    4. 당연한 재료(물, 소금 등)는 추천에서 제외
    5. 재료→상품 매핑 (참치액→참치캔)
    
    사용법:
        model = RecipeGapFillingModel.load('model.pkl')
        results = model.recommend(['양파', '마늘', '간장'])
        # 반환: [{'name': '소고기무국', 'gap_ingredients': ['대파', '무'], ...}]
    """
    
    VERSION = '2.0.0'
    
    # 제외할 키워드 (비요리 콘텐츠)
    EXCLUDE_KEYWORDS = [
        '보관법', '보관방법', '보관하는', '손질법', '손질방법', '다듬기',
        '세척법', '씻는법', '해동', '냉동보관', '냉장보관',
        '고르는법', '고르는방법', '선별법',
        '효능', '효과', '칼로리', '만들기 전', '준비하기'
    ]
    
    # 추천에서 제외할 재료 (당연히 있거나 구매할 필요 없는 것들)
    EXCLUDE_INGREDIENTS = {
        '물', '냉수', '찬물', '뜨거운물', '끓는물', '미지근한물',
        '소금', '설탕', '후추', '후춧가루', '흑후추',
        '식용유', '기름', '포도씨유', '카놀라유',
        '간장', '진간장', '국간장', '양조간장',
        '고춧가루', '고추장', '된장',
        '참기름', '들기름',
        '식초', '맛술', '청주', '미림',
        '다진마늘', '다진파', '생강가루',
        '얼음', '육수', '멸치육수', '다시마육수', '채수',
        '밥', '찬밥', '따뜻한밥',
    }
    
    # 메인 재료 카테고리 (높은 가중치)
    MAIN_INGREDIENT_CATEGORIES = {
        '육류': [
            '소고기', '돼지고기', '닭고기', '오리고기', '양고기',
            '삼겹살', '목살', '안심', '등심', '갈비', '차돌박이',
            '닭가슴살', '닭다리', '닭날개', '통닭',
            '베이컨', '햄', '소시지', '살라미',
            '불고기', '갈비살', '항정살', '가브리살', '우삼겹',
        ],
        '해산물': [
            '생선', '연어', '고등어', '갈치', '조기', '광어', '우럭', '참치',
            '오징어', '문어', '낙지', '주꾸미',
            '새우', '게', '꽃게', '대게', '랍스터',
            '조개', '바지락', '모시조개', '홍합', '전복', '굴', '가리비',
            '멸치', '북어', '황태', '대구', '명태',
            '참치캔', '고등어통조림', '연어통조림',
        ],
        '주요채소': [
            '배추', '무', '감자', '고구마', '당근', '양배추',
            '시금치', '깻잎', '상추', '청경채', '브로콜리',
            '호박', '애호박', '단호박', '가지', '오이',
            '버섯', '표고버섯', '양송이버섯', '느타리버섯', '팽이버섯',
            '두부', '콩나물', '숙주',
        ],
        '탄수화물': [
            '쌀', '현미', '잡곡',
            '면', '국수', '소면', '우동면', '라면사리', '당면', '쫄면',
            '스파게티면', '파스타면', '펜네', '마카로니',
            '떡', '떡국떡', '가래떡', '떡볶이떡',
            '빵', '식빵', '바게트',
        ],
    }
    
    # 재료 → 상품 매핑
    INGREDIENT_TO_PRODUCT_MAP = {
        '참치액': '참치캔', '멸치액젓': '멸치', '까나리액젓': '멸치', '새우젓': '새우',
        '멸치육수': '멸치', '다시마육수': '다시마', '채수': '양파',
        '사골육수': '사골', '치킨스톡': '닭고기', '비프스톡': '소고기',
        '계란물': '계란', '달걀물': '계란', '전분물': '전분', '녹말물': '전분',
        '마늘가루': '마늘', '양파가루': '양파', '생강가루': '생강',
        '통조림옥수수': '옥수수', '냉동새우': '새우', '냉동오징어': '오징어',
        '다진돼지고기': '돼지고기', '다진소고기': '소고기',
    }
    
    def __init__(self,
                 recipe_ingredient_sets: Dict[int, Set[str]],
                 ingredient_to_recipes: Dict[str, List[int]],
                 recipe_metadata: Dict[int, dict],
                 synonym_dict: Dict[str, List[str]],
                 params: Optional[Dict] = None,
                 valid_recipe_ids: Optional[Set[int]] = None):
        self.recipe_ingredient_sets = recipe_ingredient_sets
        self.ingredient_to_recipes = ingredient_to_recipes
        self.recipe_metadata = recipe_metadata
        self.synonym_dict = synonym_dict
        
        # 역방향 동의어 매핑
        self.synonym_to_standard = {}
        for standard, synonyms in synonym_dict.items():
            for syn in synonyms:
                self.synonym_to_standard[syn] = standard
        
        # 메인 재료 집합 구축
        self.main_ingredients = set()
        for category_items in self.MAIN_INGREDIENT_CATEGORIES.values():
            self.main_ingredients.update(category_items)
        
        # 기본 파라미터 (v2.0)
        self.params = params or {
            'min_match_ratio': 0.3,
            'max_gap_count': 5,
            'popularity_weight': 0.4,
            'match_weight': 0.4,
            'gap_penalty_weight': 0.2,
            'main_ingredient_bonus': 0.3,
            'min_ingredients': 3,
            'min_view_count': 500,
        }
        
        # 유효 레시피 ID
        if valid_recipe_ids is not None:
            self.valid_recipe_ids = valid_recipe_ids
        else:
            self.valid_recipe_ids = self._filter_valid_recipes()
        
        # 최대 인기도 캐싱
        self._max_popularity = max(
            (m.get('popularity_score', 0) or 0 for m in recipe_metadata.values()),
            default=1
        )
    
    def _filter_valid_recipes(self) -> Set[int]:
        """유효한 레시피만 필터링"""
        valid_ids = set()
        excluded = {'ingredients': 0, 'keyword': 0, 'view_count': 0}
        
        for recipe_id, ingredients in self.recipe_ingredient_sets.items():
            metadata = self.recipe_metadata.get(recipe_id, {})
            title = metadata.get('title', '') or metadata.get('name', '')
            view_count = metadata.get('view_count', 0) or 0
            
            if len(ingredients) < self.params.get('min_ingredients', 3):
                excluded['ingredients'] += 1
                continue
            
            if any(kw in title for kw in self.EXCLUDE_KEYWORDS):
                excluded['keyword'] += 1
                continue
            
            if view_count < self.params.get('min_view_count', 500):
                excluded['view_count'] += 1
                continue
            
            valid_ids.add(recipe_id)
        
        print(f"  유효 레시피: {len(valid_ids):,}개 / {len(self.recipe_ingredient_sets):,}개")
        print(f"  제외: 재료부족 {excluded['ingredients']:,}, 비요리 {excluded['keyword']:,}, 저조회 {excluded['view_count']:,}")
        
        return valid_ids
    
    @classmethod
    def load(cls, path: str) -> 'RecipeGapFillingModel':
        """피클 파일에서 모델 로드"""
        with open(path, 'rb') as f:
            data = pickle.load(f)
        
        return cls(
            recipe_ingredient_sets=data['recipe_ingredient_sets'],
            ingredient_to_recipes=data['ingredient_to_recipes'],
            recipe_metadata=data['recipe_metadata'],
            synonym_dict=data['synonym_dict'],
            params=data.get('params'),
            valid_recipe_ids=data.get('valid_recipe_ids')
        )
    
    def save(self, path: str) -> None:
        """모델을 피클 파일로 저장"""
        data = {
            'recipe_ingredient_sets': self.recipe_ingredient_sets,
            'ingredient_to_recipes': self.ingredient_to_recipes,
            'recipe_metadata': self.recipe_metadata,
            'synonym_dict': self.synonym_dict,
            'params': self.params,
            'valid_recipe_ids': self.valid_recipe_ids,
            'version': self.VERSION,
            'created_at': datetime.now().isoformat()
        }
        
        with open(path, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    def normalize_ingredient(self, name: str) -> str:
        if not name:
            return name
        name = str(name).strip()
        if name in self.synonym_to_standard:
            return self.synonym_to_standard[name]
        for standard in self.synonym_dict.keys():
            if standard in name:
                return standard
        return name
    
    def map_to_product(self, ingredient: str) -> str:
        """재료명을 실제 구매 가능한 상품명으로 매핑"""
        return self.INGREDIENT_TO_PRODUCT_MAP.get(ingredient, ingredient)
    
    def is_main_ingredient(self, ingredient: str) -> bool:
        return ingredient in self.main_ingredients
    
    def should_exclude_from_gap(self, ingredient: str) -> bool:
        return ingredient in self.EXCLUDE_INGREDIENTS
    
    def get_ingredient_weight(self, ingredient: str) -> float:
        if self.should_exclude_from_gap(ingredient):
            return 0.0
        if self.is_main_ingredient(ingredient):
            return 1.5
        seasoning_keywords = ['가루', '소스', '장', '젓', '액', '분', '즙']
        if any(kw in ingredient for kw in seasoning_keywords):
            return 0.7
        return 1.0
    
    def recommend(self,
                  cart_ingredients: List[str],
                  top_k: int = 10,
                  min_match_ratio: Optional[float] = None,
                  max_gap_count: Optional[int] = None) -> List[Dict]:
        """
        장바구니 기반 레시피 추천 (v2.0)
        
        Returns:
            List[Dict]: 추천 레시피 목록
                - name: 요리명 (CKG_NM)
                - gap_ingredients: 상품 매핑된 Gap 재료
        """
        if not cart_ingredients:
            return []
        
        min_match_ratio = min_match_ratio or self.params['min_match_ratio']
        max_gap_count = max_gap_count or self.params['max_gap_count']
        
        cart_set = set(self.normalize_ingredient(ing) for ing in cart_ingredients)
        
        # 후보 레시피 검색
        candidates = set()
        for ingredient in cart_ingredients:
            normalized = self.normalize_ingredient(ingredient)
            if normalized in self.ingredient_to_recipes:
                for recipe_id in self.ingredient_to_recipes[normalized]:
                    if recipe_id in self.valid_recipe_ids:
                        candidates.add(recipe_id)
        
        if not candidates:
            return []
        
        results = []
        for recipe_id in candidates:
            recipe_ingredients = self.recipe_ingredient_sets.get(recipe_id, set())
            if not recipe_ingredients:
                continue
            
            matched = cart_set & recipe_ingredients
            gaps = recipe_ingredients - cart_set
            
            # 제외 재료 필터링 및 상품 매핑
            filtered_gaps = [g for g in gaps if not self.should_exclude_from_gap(g)]
            mapped_gaps = list(dict.fromkeys([self.map_to_product(g) for g in filtered_gaps]))
            
            match_ratio = len(matched) / len(recipe_ingredients)
            
            if match_ratio < min_match_ratio:
                continue
            if len(mapped_gaps) > max_gap_count:
                continue
            
            metadata = self.recipe_metadata.get(recipe_id, {})
            
            # 메인 재료 보너스
            main_matched = sum(1 for m in matched if self.is_main_ingredient(m))
            main_bonus = main_matched * self.params['main_ingredient_bonus'] / max(len(recipe_ingredients), 1)
            
            # Gap 가중치
            gap_weighted = sum(self.get_ingredient_weight(g) for g in filtered_gaps)
            
            # 인기도 정규화
            popularity = metadata.get('popularity_score', 0) or 0
            norm_popularity = popularity / self._max_popularity if self._max_popularity > 0 else 0
            
            # 최종 점수
            gap_penalty = min(gap_weighted / (self.params['max_gap_count'] * 1.5), 1.0)
            final_score = (
                self.params['match_weight'] * match_ratio +
                self.params['popularity_weight'] * norm_popularity +
                main_bonus -
                self.params['gap_penalty_weight'] * gap_penalty
            )
            final_score = max(0, min(1, final_score))
            
            results.append({
                'recipe_id': recipe_id,
                'name': metadata.get('name', ''),           # ★ 요리명 (CKG_NM)
                'title': metadata.get('title', ''),
                'matched_ingredients': list(matched),
                'gap_ingredients': mapped_gaps,              # ★ 상품 매핑됨
                'match_ratio': match_ratio,
                'gap_count': len(mapped_gaps),
                'total_ingredients': len(recipe_ingredients),
                'view_count': metadata.get('view_count', 0),
                'final_score': final_score,
                'category': metadata.get('category', ''),
                'difficulty': metadata.get('difficulty', '')
            })
        
        results.sort(key=lambda x: x['final_score'], reverse=True)
        return results[:top_k]
    
    def get_gap_recommendations(self,
                                 cart_ingredients: List[str],
                                 top_k_recipes: int = 5,
                                 top_k_gaps: int = 10) -> Dict:
        """Gap 기반 추가 재료 추천 (메인 재료 우선)"""
        recommendations = self.recommend(cart_ingredients, top_k=top_k_recipes)
        
        if not recommendations:
            return {'gap_ingredients': [], 'recipes': [], 'message': '매칭되는 레시피가 없습니다.'}
        
        gap_scores = defaultdict(lambda: {'frequency': 0, 'weight': 0, 'recipes': []})
        
        for rec in recommendations:
            for gap in rec['gap_ingredients']:
                weight = self.get_ingredient_weight(gap)
                gap_scores[gap]['frequency'] += 1
                gap_scores[gap]['weight'] += weight * rec['final_score']
                gap_scores[gap]['recipes'].append({
                    'recipe_id': rec['recipe_id'],
                    'name': rec['name'],
                    'match_ratio': rec['match_ratio']
                })
        
        sorted_gaps = sorted(gap_scores.items(), key=lambda x: x[1]['weight'], reverse=True)[:top_k_gaps]
        
        gap_recommendations = []
        for gap_name, gap_info in sorted_gaps:
            gap_recommendations.append({
                'ingredient': gap_name,
                'is_main_ingredient': self.is_main_ingredient(gap_name),
                'frequency': gap_info['frequency'],
                'score': round(gap_info['weight'], 3),
                'related_recipes': gap_info['recipes'][:3]
            })
        
        return {
            'gap_ingredients': gap_recommendations,
            'recipes': recommendations,
            'total_recipes_analyzed': len(recommendations)
        }
    
    def get_model_info(self) -> Dict:
        return {
            'version': self.VERSION,
            'num_recipes': len(self.recipe_ingredient_sets),
            'num_valid_recipes': len(self.valid_recipe_ids),
            'num_ingredients': len(self.ingredient_to_recipes),
            'num_synonyms': len(self.synonym_dict),
            'num_main_ingredients': len(self.main_ingredients),
            'num_excluded_ingredients': len(self.EXCLUDE_INGREDIENTS),
            'params': self.params
        }

print("RecipeGapFillingModel v2.0 클래스 정의 완료 (프로덕션용)")

RecipeGapFillingModel v2.0 클래스 정의 완료 (프로덕션용)


## 5.2 기존 데이터 로드 및 모델 생성

In [12]:
# 기존 처리된 데이터 로드
PROCESSED_DIR = '../data/processed'

# 이전 단계에서 저장한 데이터 로드
with open(f'{PROCESSED_DIR}/gapfilling_model.pkl', 'rb') as f:
    existing_data = pickle.load(f)

print(f"기존 모델 데이터 로드 완료")
print(f"  - 레시피 수: {len(existing_data['recipe_ingredient_sets']):,}")
print(f"  - 재료 수: {len(existing_data['ingredient_to_recipes']):,}")

기존 모델 데이터 로드 완료
  - 레시피 수: 23,191
  - 재료 수: 9,147


In [13]:
# 프로덕션 모델 생성
model = RecipeGapFillingModel(
    recipe_ingredient_sets=existing_data['recipe_ingredient_sets'],
    ingredient_to_recipes=existing_data['ingredient_to_recipes'],
    recipe_metadata=existing_data['recipe_metadata'],
    synonym_dict=existing_data['synonym_dict'],
    params=existing_data.get('params')
)

print("모델 정보:")
for k, v in model.get_model_info().items():
    print(f"  {k}: {v}")

  유효 레시피: 9,174개 / 23,191개
  제외: 재료부족 678, 비요리 325, 저조회 13,014
모델 정보:
  version: 2.0.0
  num_recipes: 23191
  num_valid_recipes: 9174
  num_ingredients: 9147
  num_synonyms: 23
  num_main_ingredients: 101
  num_excluded_ingredients: 39
  params: {'min_match_ratio': 0.3, 'max_gap_count': 5, 'popularity_weight': 0.3, 'match_weight': 0.5, 'gap_penalty_weight': 0.2, 'main_ingredient_bonus': 0.3, 'min_ingredients': 3, 'min_view_count': 500}


## 5.3 프로덕션 피클 파일 생성

In [14]:
# 피클 파일 저장 경로
PROD_MODEL_DIR = '../pred/models'
os.makedirs(PROD_MODEL_DIR, exist_ok=True)

# 프로덕션 모델 저장
PROD_MODEL_PATH = f'{PROD_MODEL_DIR}/recipe_gapfilling_v1.pkl'
model.save(PROD_MODEL_PATH)

# 파일 크기 확인
file_size = os.path.getsize(PROD_MODEL_PATH) / (1024 * 1024)
print(f"프로덕션 모델 저장 완료: {PROD_MODEL_PATH}")
print(f"파일 크기: {file_size:.2f} MB")

프로덕션 모델 저장 완료: ../pred/models/recipe_gapfilling_v1.pkl
파일 크기: 5.96 MB


In [15]:
# 모델 로딩 테스트
loaded_model = RecipeGapFillingModel.load(PROD_MODEL_PATH)

print("로드된 모델 정보:")
for k, v in loaded_model.get_model_info().items():
    print(f"  {k}: {v}")

로드된 모델 정보:
  version: 2.0.0
  num_recipes: 23191
  num_valid_recipes: 9174
  num_ingredients: 9147
  num_synonyms: 23
  num_main_ingredients: 101
  num_excluded_ingredients: 39
  params: {'min_match_ratio': 0.3, 'max_gap_count': 5, 'popularity_weight': 0.3, 'match_weight': 0.5, 'gap_penalty_weight': 0.2, 'main_ingredient_bonus': 0.3, 'min_ingredients': 3, 'min_view_count': 500}


## 5.4 통합 테스트

In [16]:
# 테스트 케이스 정의
test_scenarios = [
    {
        'name': '한식 기본 재료',
        'cart': ['양파', '마늘', '간장', '고춧가루', '참기름'],
        'expected_categories': ['한식', '반찬']
    },
    {
        'name': '이탈리안 재료',
        'cart': ['토마토', '올리브유', '마늘', '바질'],
        'expected_categories': ['양식', '파스타']
    },
    {
        'name': '떡국 재료',
        'cart': ['떡국떡', '계란', '대파'],
        'expected_categories': ['한식', '국/탕']
    },
    {
        'name': '볶음 요리 재료',
        'cart': ['돼지고기', '양파', '당근', '간장'],
        'expected_categories': ['한식', '볶음']
    },
    {
        'name': '국물 요리 재료',
        'cart': ['두부', '된장', '호박', '파'],
        'expected_categories': ['한식', '국/찌개']
    }
]

print(f"정의된 테스트 시나리오: {len(test_scenarios)}개")

정의된 테스트 시나리오: 5개


In [17]:
# 통합 테스트 실행
print("=" * 80)
print("통합 테스트 실행")
print("=" * 80)

test_results = []

for scenario in test_scenarios:
    print(f"\n[{scenario['name']}]")
    print(f"장바구니: {scenario['cart']}")
    
    # 추천 실행
    start_time = time.time()
    recommendations = loaded_model.recommend(scenario['cart'], top_k=5)
    elapsed = (time.time() - start_time) * 1000  # ms
    
    print(f"응답 시간: {elapsed:.2f}ms")
    print(f"추천 레시피 수: {len(recommendations)}개")
    
    if recommendations:
        print(f"\n추천 결과:")
        for i, rec in enumerate(recommendations[:3], 1):
            print(f"  {i}. {rec['title']}")
            print(f"     매칭: {rec['match_ratio']*100:.0f}% | Gap: {rec['gap_count']}개 | 점수: {rec['final_score']:.3f}")
            print(f"     부족 재료: {rec['gap_ingredients'][:3]}..." if len(rec['gap_ingredients']) > 3 else f"     부족 재료: {rec['gap_ingredients']}")
    
    test_results.append({
        'scenario': scenario['name'],
        'cart_size': len(scenario['cart']),
        'num_recommendations': len(recommendations),
        'response_time_ms': elapsed,
        'success': len(recommendations) > 0
    })

통합 테스트 실행

[한식 기본 재료]
장바구니: ['양파', '마늘', '간장', '고춧가루', '참기름']
응답 시간: 26.04ms
추천 레시피 수: 5개

추천 결과:
  1. 양파요리 양파장아찌 담그는법 만들기 
     매칭: 40% | Gap: 1개 | 점수: 0.380
     부족 재료: ['고추']
  2. 양념간장 만드는 법 간장 양념장 만들기
     매칭: 60% | Gap: 2개 | 점수: 0.358
     부족 재료: ['깨', '대파']
  3. 마늘쫑 장아찌 무침 만드는법
     매칭: 75% | Gap: 1개 | 점수: 0.354
     부족 재료: ['참깨']

[이탈리안 재료]
장바구니: ['토마토', '올리브유', '마늘', '바질']
응답 시간: 13.52ms
추천 레시피 수: 5개

추천 결과:
  1. 편스토랑 류수영 마늘수육 냄비수육 레시피 통삼겹살 바베큐 st 레시피 무수분 수육 (+ 조리시간 25분)
     매칭: 50% | Gap: 1개 | 점수: 0.250
     부족 재료: ['통삼겹']
  2. 토마토쥬스 여름음료 토마토주스 만들기 토마토 데치기 토마토요리 레시피
     매칭: 50% | Gap: 1개 | 점수: 0.244
     부족 재료: ['레몬즙']
  3. 마늘튀김 요리 
     매칭: 50% | Gap: 1개 | 점수: 0.236
     부족 재료: ['파슬리가루']

[떡국 재료]
장바구니: ['떡국떡', '계란', '대파']
응답 시간: 11.01ms
추천 레시피 수: 5개

추천 결과:
  1. 떡라면 끓이는법 계란라면 떡국떡 요리
     매칭: 60% | Gap: 1개 | 점수: 0.339
     부족 재료: ['라면']
  2. 계란찜 만들기/ 전자레인지 계란찜/ 푸딩 계란찜
     매칭: 67% | Gap: 0개 | 점수: 0.336
     부족 재료: []
  3. 설날음식 떡국끓이기 
     매칭: 60% | Gap: 1개 | 점수: 0.325
     부

In [18]:
# Gap 추천 테스트
print("\n" + "=" * 80)
print("Gap 추천 테스트")
print("=" * 80)

test_cart = ['양파', '마늘', '간장', '소고기']
print(f"\n장바구니: {test_cart}")

gap_results = loaded_model.get_gap_recommendations(test_cart, top_k_recipes=10, top_k_gaps=5)

print(f"\n분석된 레시피: {gap_results['total_recipes_analyzed']}개")
print(f"\n추천 추가 재료 (Gap):")

for i, gap in enumerate(gap_results['gap_ingredients'], 1):
    main_tag = "🥩 메인" if gap['is_main_ingredient'] else "   일반"
    print(f"  {i}. [{main_tag}] {gap['ingredient']} (빈도: {gap['frequency']}회, 점수: {gap['score']:.3f})")
    for recipe in gap['related_recipes'][:2]:
        print(f"     → {recipe['name']} (매칭 {recipe['match_ratio']*100:.0f}%)")  # ★ title → name


Gap 추천 테스트

장바구니: ['양파', '마늘', '간장', '소고기']

분석된 레시피: 10개

추천 추가 재료 (Gap):
  1. [🥩 메인] 참치캔 (빈도: 1회, 점수: 0.577)
     → 소고기미역국 (매칭 43%)
  2. [   일반] 자른미역 (빈도: 2회, 점수: 0.569)
     → 소고기미역국 (매칭 57%)
     → 소고기미역국 (매칭 50%)
  3. [   일반] 올리고당 (빈도: 1회, 점수: 0.419)
     → 소고기장조림 (매칭 67%)
  4. [   일반] 미역 (빈도: 1회, 점수: 0.384)
     → 소고기미역국 (매칭 43%)
  5. [   일반] 고추 (빈도: 1회, 점수: 0.380)
     → 양파장아찌 (매칭 40%)


## 5.5 성능 벤치마크

In [19]:
import random

def benchmark_model(model, n_iterations=100):
    """
    모델 성능 벤치마크
    
    Args:
        model: GapFilling 모델
        n_iterations: 테스트 반복 횟수
    
    Returns:
        Dict: 벤치마크 결과
    """
    # 샘플 재료 목록
    sample_ingredients = list(model.ingredient_to_recipes.keys())[:100]
    
    response_times = []
    recommendation_counts = []
    
    for _ in range(n_iterations):
        # 랜덤 장바구니 생성 (3-7개 재료)
        cart_size = random.randint(3, 7)
        cart = random.sample(sample_ingredients, min(cart_size, len(sample_ingredients)))
        
        # 추천 실행 및 시간 측정
        start_time = time.time()
        recommendations = model.recommend(cart, top_k=10)
        elapsed = (time.time() - start_time) * 1000
        
        response_times.append(elapsed)
        recommendation_counts.append(len(recommendations))
    
    return {
        'iterations': n_iterations,
        'avg_response_time_ms': np.mean(response_times),
        'p50_response_time_ms': np.percentile(response_times, 50),
        'p95_response_time_ms': np.percentile(response_times, 95),
        'p99_response_time_ms': np.percentile(response_times, 99),
        'max_response_time_ms': np.max(response_times),
        'avg_recommendations': np.mean(recommendation_counts),
        'success_rate': sum(1 for c in recommendation_counts if c > 0) / n_iterations
    }

# 벤치마크 실행
print("성능 벤치마크 실행 중...")
benchmark_results = benchmark_model(loaded_model, n_iterations=200)

print("\n" + "=" * 60)
print("성능 벤치마크 결과")
print("=" * 60)
for metric, value in benchmark_results.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.2f}")
    else:
        print(f"{metric}: {value}")

성능 벤치마크 실행 중...

성능 벤치마크 결과
iterations: 200
avg_response_time_ms: 5.90
p50_response_time_ms: 5.51
p95_response_time_ms: 13.52
p99_response_time_ms: 16.04
max_response_time_ms: 18.53
avg_recommendations: 7.97
success_rate: 0.94


## 5.6 API 통합용 헬퍼 함수

In [20]:
# API 통합을 위한 헬퍼 함수들

def api_recommend_recipes(cart_items: List[str], top_k: int = 10) -> Dict:
    """
    API용 레시피 추천 함수
    
    Args:
        cart_items: 장바구니 상품명 리스트
        top_k: 추천할 레시피 개수
    
    Returns:
        Dict: API 응답 형식의 추천 결과
    """
    if not cart_items:
        return {
            'success': False,
            'error': '장바구니가 비어있습니다.',
            'recommendations': []
        }
    
    try:
        recommendations = loaded_model.recommend(cart_items, top_k=top_k)
        
        return {
            'success': True,
            'cart_items': cart_items,
            'total_recommendations': len(recommendations),
            'recommendations': [
                {
                    'recipe_id': rec['recipe_id'],
                    'name': rec['name'],              # ★ 요리명
                    'title': rec['title'],
                    'match_ratio': round(rec['match_ratio'], 3),
                    'gap_ingredients': rec['gap_ingredients'],
                    'score': round(rec['final_score'], 3)
                }
                for rec in recommendations
            ]
        }
    except Exception as e:
        return {
            'success': False,
            'error': str(e),
            'recommendations': []
        }


def api_get_gap_suggestions(cart_items: List[str], top_k: int = 5) -> Dict:
    """
    API용 Gap 재료 추천 함수
    
    Args:
        cart_items: 장바구니 상품명 리스트
        top_k: 추천할 Gap 재료 개수
    
    Returns:
        Dict: API 응답 형식의 Gap 추천 결과
    """
    if not cart_items:
        return {
            'success': False,
            'error': '장바구니가 비어있습니다.',
            'suggestions': []
        }
    
    try:
        results = loaded_model.get_gap_recommendations(
            cart_items, 
            top_k_recipes=10, 
            top_k_gaps=top_k
        )
        
        return {
            'success': True,
            'cart_items': cart_items,
            'suggestions': [
                {
                    'ingredient': gap['ingredient'],
                    'is_main_ingredient': gap['is_main_ingredient'],
                    'relevance': gap['frequency'],
                    'score': gap['score'],
                    'related_recipes': [r['name'] for r in gap['related_recipes']]  # ★ title → name
                }
                for gap in results['gap_ingredients']
            ]
        }
    except Exception as e:
        return {
            'success': False,
            'error': str(e),
            'suggestions': []
        }

print("API 헬퍼 함수 정의 완료")

API 헬퍼 함수 정의 완료


In [21]:
# API 함수 테스트
print("API 함수 테스트:")
print("=" * 60)

# 레시피 추천 API 테스트
result = api_recommend_recipes(['양파', '마늘', '간장'])
print("\napi_recommend_recipes 결과:")
print(json.dumps(result, ensure_ascii=False, indent=2)[:500] + "...")

# Gap 추천 API 테스트
result = api_get_gap_suggestions(['양파', '마늘', '간장'])
print("\napi_get_gap_suggestions 결과:")
print(json.dumps(result, ensure_ascii=False, indent=2))

API 함수 테스트:

api_recommend_recipes 결과:
{
  "success": true,
  "cart_items": [
    "양파",
    "마늘",
    "간장"
  ],
  "total_recommendations": 10,
  "recommendations": [
    {
      "recipe_id": 7029816,
      "name": "양파장아찌",
      "title": "양파요리 양파장아찌 담그는법 만들기 ",
      "match_ratio": 0.4,
      "gap_ingredients": [
        "고추"
      ],
      "score": 0.38
    },
    {
      "recipe_id": 7026696,
      "name": "양파볶음",
      "title": "간단한 반찬 양파볶음 양파 요리 간장양파볶음",
      "match_ratio": 0.5,
      "gap_ingredients": [
        "통깨"
      ],
 ...

api_get_gap_suggestions 결과:
{
  "success": true,
  "cart_items": [
    "양파",
    "마늘",
    "간장"
  ],
  "suggestions": [
    {
      "ingredient": "소주",
      "is_main_ingredient": false,
      "relevance": 2,
      "score": 0.438,
      "related_recipes": [
        "마늘장아찌",
        "간장마늘쫑장아찌"
      ]
    },
    {
      "ingredient": "고추",
      "is_main_ingredient": false,
      "relevance": 1,
      "score": 0.38,
      "related_recipes": [
        "양

## 5.7 최종 결과 저장

In [22]:
# 테스트 결과 저장
test_report = {
    'model_info': loaded_model.get_model_info(),
    'test_scenarios': test_results,
    'benchmark': benchmark_results,
    'model_path': PROD_MODEL_PATH,
    'created_at': datetime.now().isoformat()
}

with open(f'{PROCESSED_DIR}/integration_test_report.json', 'w', encoding='utf-8') as f:
    json.dump(test_report, f, ensure_ascii=False, indent=2, default=str)

print(f"테스트 리포트 저장: {PROCESSED_DIR}/integration_test_report.json")

테스트 리포트 저장: ../data/processed/integration_test_report.json


In [23]:
# 최종 요약
print("\n" + "=" * 80)
print("GapFilling 레시피 추천 모델 - 최종 요약")
print("=" * 80)

print(f"\n[모델 정보]")
print(f"  버전: {loaded_model.VERSION}")
print(f"  레시피 수: {len(loaded_model.recipe_ingredient_sets):,}개")
print(f"  재료 수: {len(loaded_model.ingredient_to_recipes):,}개")

print(f"\n[성능]")
print(f"  평균 응답 시간: {benchmark_results['avg_response_time_ms']:.2f}ms")
print(f"  P95 응답 시간: {benchmark_results['p95_response_time_ms']:.2f}ms")
print(f"  성공률: {benchmark_results['success_rate']*100:.1f}%")

print(f"\n[파일 위치]")
print(f"  프로덕션 모델: {PROD_MODEL_PATH}")
print(f"  테스트 리포트: {PROCESSED_DIR}/integration_test_report.json")

print(f"\n[사용 방법]")
print("  from recipe_model import RecipeGapFillingModel")
print(f"  model = RecipeGapFillingModel.load('{PROD_MODEL_PATH}')")
print("  results = model.recommend(['양파', '마늘', '간장'])")

print("\n✓ 모든 단계 완료!")


GapFilling 레시피 추천 모델 - 최종 요약

[모델 정보]
  버전: 2.0.0
  레시피 수: 23,191개
  재료 수: 9,147개

[성능]
  평균 응답 시간: 5.90ms
  P95 응답 시간: 13.52ms
  성공률: 94.0%

[파일 위치]
  프로덕션 모델: ../pred/models/recipe_gapfilling_v1.pkl
  테스트 리포트: ../data/processed/integration_test_report.json

[사용 방법]
  from recipe_model import RecipeGapFillingModel
  model = RecipeGapFillingModel.load('../pred/models/recipe_gapfilling_v1.pkl')
  results = model.recommend(['양파', '마늘', '간장'])

✓ 모든 단계 완료!
